# 이탈 예측 모델 앙상블 (4) - XGBoost + LightGBM

이 노트북은 구독자 10만~100만 사이의 유튜버 데이터를 활용하여, 유튜버가 향후 이탈할 것인지 예측하기 위한 앙상블 모델입니다.

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
)
from scipy.stats import linregress
from functools import reduce
import warnings

warnings.filterwarnings('ignore')
available_fonts = {font.name for font in fm.fontManager.ttflist}
if 'Malgun Gothic' in available_fonts:
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif 'AppleGothic' in available_fonts:
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False


In [2]:
# 데이터 로드
wide_df = pd.read_csv('../../data/raw/filtered_dataset_wide.csv')
long_df = pd.read_csv('../../data/raw/filtered_dataset_long.csv')
view_volatility_df = pd.read_csv('../../data/raw/view_volatility_output.csv')
upload_regularity_df = pd.read_csv('../../data/raw/upload_regularity_score.csv')
active_viewer_df = pd.read_csv('../../data/raw/active_viewer_score.csv')
sensitive_keyword_df = pd.read_csv('../../data/raw/sensitive_keyword_score.csv')

print(f"Wide dataset shape: {wide_df.shape}")
print(f"Long dataset shape: {long_df.shape}")

Wide dataset shape: (3336, 22)
Long dataset shape: (166449, 18)


## 2. 데이터 전처리 및 라벨링

마지막 업로드로부터 180일 이상 경과한 채널을 '이탈(1)'로 정의합니다.

In [3]:
CHURN_THRESHOLD = 180
wide_df['is_churned'] = (wide_df['days_since_last_upload'] >= CHURN_THRESHOLD).astype(int)


print("이탈 여부 분포 :")
print(wide_df['is_churned'].value_counts())
print(f"이탈 비율 : {wide_df['is_churned'].mean():.2%}")

이탈 여부 분포 :
is_churned
0    2641
1     695
Name: count, dtype: int64
이탈 비율 : 20.83%


## 3. 피처 엔지니어링

In [4]:
long_df['published_at'] = pd.to_datetime(long_df['published_at'])
NOW = long_df['published_at'].max()

def extract_long_features(group):
    g = group.sort_values("published_at")

    # 1. upload_slope: 월별 업로드 수 선형 기울기
    g["ym"] = g["published_at"].dt.to_period("M")
    monthly = g.groupby("ym").size().reset_index(name="cnt")
    if len(monthly) >= 3:
        slope, *_ = linregress(range(len(monthly)), monthly["cnt"])
    else:
        slope = np.nan

    # 2. recent_3m_upload_count
    t3 = NOW - pd.Timedelta(days=90)
    recent_3m = (g["published_at"] >= t3).sum()

    # 3. recent_view_ratio (최신 10개 평균 / 전체 평균)
    views = g["view_count"].dropna().tolist()
    if len(views) >= 10:
        recent_avg = np.mean(views[-10:])
        total_avg  = np.mean(views)
        view_ratio = round(recent_avg / (total_avg + 1e-9), 4)
    else:
        view_ratio = np.nan

    # 4. days_between_last2: 마지막 두 영상 사이 간격
    dates = g["published_at"].dropna().tolist()
    if len(dates) >= 2:
        days_last2 = (dates[-1] - dates[-2]).days
    else:
        days_last2 = np.nan

    return pd.Series({
        "upload_slope": slope,
        "recent_3m_upload_count": int(recent_3m),
        "recent_view_ratio": view_ratio,
        "days_between_last2": days_last2,
    })

long_features = long_df.groupby("channel_id").apply(extract_long_features).reset_index()

### 피처(Feature) 설명

#### WIDE_FEATURES — 채널 단위 집계 피처 (`wide_df`)

**채널 규모 / 누적 지표**
- `subscriber_count` : 구독자 수
- `view_count` : 채널 누적 조회수
- `total_video_count` : 채널 전체 영상 수

**업로드 패턴 (규칙성)**
- `avg_upload_interval_days` : 평균 업로드 간격(일)
- `std_upload_interval_days` : 업로드 간격의 표준편차 (들쭉날쭉함)
- `max_gap_days` : 최대 업로드 공백 기간
- `hiatus_count_30d` : 30일 이상 쉰 횟수 (휴지기 빈도)
- ~~`upload_freq_change_rate`~~ : 업로드 빈도 변화율 *(미사용)*

**영상별 성과 / 참여도**
- `avg_view_count`, `std_view_count` : 영상당 평균 조회수와 변동성
- `avg_like_count`, `avg_comment_count` : 영상당 평균 좋아요·댓글 수
- `avg_engagement_rate` : 평균 참여율 ((좋아요+댓글)/조회수 류)
- `shorts_ratio` : 전체 영상 중 쇼츠 비중
- `avg_shorts_view`, `avg_normal_view` : 쇼츠 / 일반 영상의 평균 조회수

#### LONG_FEATURES — 시계열 파생 피처 (`extract_long_features()`로 계산)

- `upload_slope` : 월별 업로드 수의 선형회귀 기울기 (음수 → 활동 감소 추세)
- ~~`recent_3m_upload_count`~~ : 최근 90일간 업로드 수 *(미사용)*
- `recent_view_ratio` : 최근 10개 영상 평균 조회수 / 전체 평균 조회수 (1 미만이면 최근 성과 하락)
- `days_between_last2` : 마지막 두 영상 사이 간격(일)

#### 병합
`df = wide_df.merge(long_features, on="channel_id", how="left")`
→ 채널 집계 데이터(`wide_df`)에 시계열 파생 피처(`long_features`)를 `channel_id` 기준으로 left join 하여 학습용 테이블 생성.

즉, **채널의 규모·성과 + 업로드 패턴 + 최근 추세** 세 축으로 이탈 여부(`is_churned`, 180일 이상 미업로드)를 예측합니다.

In [5]:
WIDE_FEATURES = [
    "subscriber_count",            # 구독자 수
    "view_count",                  # 채널 누적 조회수
    "total_video_count",           # 채널 전체 영상 수
    "avg_upload_interval_days",    # 평균 업로드 간격(일)
    "std_upload_interval_days",    # 업로드 간격 표준편차 (들쭉날쭉함)
    "max_gap_days",                # 최대 업로드 공백 기간
    "hiatus_count_30d",            # 30일 이상 쉰 횟수 (휴지기 빈도)
   # "upload_freq_change_rate",    # 업로드 빈도 변화율 (미사용)
    "avg_view_count",              # 영상당 평균 조회수
    "std_view_count",              # 영상당 조회수 변동성
    "avg_like_count",              # 영상당 평균 좋아요 수
    "avg_comment_count",           # 영상당 평균 댓글 수
    "avg_engagement_rate",         # 평균 참여율 ((좋아요+댓글)/조회수)
    "shorts_ratio",                # 전체 영상 중 쇼츠 비중
    "avg_shorts_view",             # 쇼츠 영상 평균 조회수
    "avg_normal_view"              # 일반 영상 평균 조회수
]
LONG_FEATURES = [
    "upload_slope",                # 월별 업로드 수 선형회귀 기울기 (음수=활동 감소)
    # "recent_3m_upload_count",    # 최근 90일 업로드 수 (미사용)
    "recent_view_ratio",           # 최근 10개 평균 조회수 / 전체 평균 조회수
    "days_between_last2"           # 마지막 두 영상 사이 간격(일)
]

df = wide_df.merge(long_features, on="channel_id", how="left")


In [6]:
VIEW_VOLATILITY_FEATURES = [
    "volatility_prob"
]
UPLOAD_REGULARITY_FEATURES = [
    "regularity_score"
]
ACTIVE_VIEWER_FEATURES = [
    "active_viewer_score",
    "view_per_sub"
]
SENSITIVE_KEYWORD_FEATURES = [
    "sensitive_score"
]


In [7]:
view_volatility_df = view_volatility_df[["channel_id", "volatility_prob"]]
upload_regularity_df = upload_regularity_df[["channel_id", "regularity_score"]]
active_viewer_df = active_viewer_df[["channel_id", "active_viewer_score", "view_per_sub"]]
sensitive_keyword_df = sensitive_keyword_df[["channel_id", "sensitive_score"]]

In [8]:
dfs = [df, view_volatility_df, upload_regularity_df, active_viewer_df, sensitive_keyword_df]

# channel_id 기준으로 전부 join

merged_df = reduce(

    lambda left, right: pd.merge(left, right, on='channel_id', how='inner'),

    dfs

)

In [9]:
merged_df.head()

,channel_id,title,published_at,subscriber_count,view_count,total_video_count,source_file,collected_video_count,days_since_last_upload,avg_upload_interval_days,...,is_churned,upload_slope,recent_3m_upload_count,recent_view_ratio,days_between_last2,volatility_prob,regularity_score,active_viewer_score,view_per_sub,sensitive_score
0,UC0_yHLyKYYvG4bzylDs3pwA,2M (투엠) ENT,2009-03-16T14:52:53Z,100000,10030245,557,channels_must_3250_3299.csv,50.0,2.0,0.96,...,0,-4.000000,50.0,0.3145,0.0,0.015,0.6257,0.6068,0.035831,0.0000
1,UCDYdwDXhbVZXkdV6kZn-0IQ,심킹 (ABYSSKIMG),2012-05-01T15:10:54Z,100000,9668787,112,channels_must_3250_3299.csv,50.0,1120.0,22.61,...,1,-0.038462,0.0,0.4962,15.0,0.991,0.4077,0.6111,0.553646,0.0733
2,UCbr6AL4ayehAkXcdBM68e8A,김PD전당포,2012-01-06T08:48:32Z,100000,2156709,135,channels_must_3250_3299.csv,50.0,448.0,36.69,...,1,-0.884848,0.0,2.3545,0.0,0.049,0.3951,0.6064,0.015161,0.0200
3,UCJQDi71H00IXCQbxAtFem3Q,DC튜브,2015-12-19T23:24:19Z,100000,30385528,2043,channels_must_3250_3299.csv,50.0,14.0,14.12,...,0,-0.062338,3.0,0.5150,7.0,0.102,0.6316,0.3551,0.050204,0.0133
4,UCR33sqrqf6rlT2AlpgyKmyA,두리 원투쓰리코,2019-03-12T15:59:11Z,109000,123170488,1630,channels_must_3250_3299.csv,50.0,1.0,0.00,...,0,NaN,50.0,0.2627,0.0,0.728,1.0000,0.6044,0.191503,0.0000


In [10]:
ALL_FEATURES = (
    WIDE_FEATURES
    + LONG_FEATURES
    + VIEW_VOLATILITY_FEATURES
    + UPLOAD_REGULARITY_FEATURES
    + ACTIVE_VIEWER_FEATURES
    + SENSITIVE_KEYWORD_FEATURES
)

for feat in ALL_FEATURES:
    merged_df[feat] = merged_df[feat].fillna(merged_df[feat].median())

X = merged_df[ALL_FEATURES]
y = merged_df['is_churned']

print(f"Feature count: {len(ALL_FEATURES)}")
print(ALL_FEATURES)


Feature count: 23
['subscriber_count', 'view_count', 'total_video_count', 'avg_upload_interval_days', 'std_upload_interval_days', 'max_gap_days', 'hiatus_count_30d', 'avg_view_count', 'std_view_count', 'avg_like_count', 'avg_comment_count', 'avg_engagement_rate', 'shorts_ratio', 'avg_shorts_view', 'avg_normal_view', 'upload_slope', 'recent_view_ratio', 'days_between_last2', 'volatility_prob', 'regularity_score', 'active_viewer_score', 'view_per_sub', 'sensitive_score']


In [11]:
RANDOM_STATE = 42
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
model_results = {}


def make_train_valid_test_split(test_size=0.2, valid_size=0.25):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full,
        y_train_full,
        test_size=valid_size,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )
    return X_train, X_valid, X_test, y_train, y_valid, y_test


def score_classifier(model, X_data):
    return model.predict_proba(X_data)[:, 1]


def score_regressor(model, X_data):
    return np.clip(model.predict(X_data), 0, 1)


def calculate_binary_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "log_loss": log_loss(y_true, y_score),
    }


def find_best_threshold(y_true, y_score):
    best_metrics = None
    best_key = None

    for threshold in THRESHOLD_GRID:
        metrics = calculate_binary_metrics(y_true, y_score, threshold)
        key = (metrics["accuracy"], metrics["roc_auc"], metrics["f1"], metrics["recall"])
        if best_key is None or key > best_key:
            best_key = key
            best_metrics = metrics

    return best_metrics


def show_table(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def tune_model_candidates(model_name, candidates, score_func):
    rows = []
    best = None

    for idx, model_candidate in enumerate(candidates, start=1):
        model_candidate.fit(X_train, y_train)
        valid_score = np.clip(score_func(model_candidate, X_valid), 0, 1)
        valid_metrics = find_best_threshold(y_valid, valid_score)
        row = {"candidate": idx, **valid_metrics}
        rows.append(row)

        key = (valid_metrics["accuracy"], valid_metrics["roc_auc"], valid_metrics["f1"], valid_metrics["recall"])
        if best is None or key > best["key"]:
            best = {
                "key": key,
                "candidate": idx,
                "model": model_candidate,
                "threshold": valid_metrics["threshold"],
                "valid_metrics": valid_metrics,
                "score_func": score_func,
            }

    summary = pd.DataFrame(rows).sort_values(
        ["accuracy", "roc_auc", "f1", "recall"], ascending=False
    ).reset_index(drop=True)
    print(f"=== {model_name}: validation tuning summary ===")
    show_table(summary)

    return best


def evaluate_tuned_model(model_name, result):
    y_score = np.clip(result["score_func"](result["model"], X_test), 0, 1)
    threshold = result["threshold"]
    y_pred = (y_score >= threshold).astype(int)
    test_metrics = calculate_binary_metrics(y_test, y_score, threshold)

    print(f"=== {model_name}: test metrics ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print(pd.Series(test_metrics).to_string())

    model_results[model_name] = {
        "candidate": result["candidate"],
        "threshold": threshold,
        "valid_metrics": result["valid_metrics"],
        "test_metrics": test_metrics,
    }
    return y_pred, y_score, test_metrics


X_train, X_valid, X_test, y_train, y_valid, y_test = make_train_valid_test_split()
print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")


Train: (1999, 23), Valid: (667, 23), Test: (667, 23)


In [12]:
print("Training XGBoost...")
xgb_model = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_prob_valid = xgb_model.predict_proba(X_valid)[:, 1]
xgb_prob_test = xgb_model.predict_proba(X_test)[:, 1]
print("Training LightGBM...")
lgbm_model = LGBMClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1)
lgbm_model.fit(X_train, y_train)
lgbm_prob_valid = lgbm_model.predict_proba(X_valid)[:, 1]
lgbm_prob_test = lgbm_model.predict_proba(X_test)[:, 1]


Training XGBoost...
Training LightGBM...


In [13]:
# Ensemble Weights: {'xgb': 0.4, 'lgbm': 0.6}
ensemble_prob_valid = np.zeros_like(y_valid, dtype=float)
ensemble_prob_test = np.zeros_like(y_test, dtype=float)
ensemble_prob_valid += xgb_prob_valid * 0.4
ensemble_prob_test += xgb_prob_test * 0.4
ensemble_prob_valid += lgbm_prob_valid * 0.6
ensemble_prob_test += lgbm_prob_test * 0.6

best_metrics = find_best_threshold(y_valid, ensemble_prob_valid)
threshold = best_metrics["threshold"]

print("=== Ensemble Validation Metrics ===")
for k, v in best_metrics.items():
    print(f"{k}: {v:.4f}")

ensemble_pred = (ensemble_prob_test >= threshold).astype(int)
test_metrics = calculate_binary_metrics(y_test, ensemble_prob_test, threshold)

print("\n=== Ensemble Test Metrics ===")
print(classification_report(y_test, ensemble_pred, zero_division=0))
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


=== Ensemble Validation Metrics ===
threshold: 0.6000
accuracy: 0.8321
precision: 0.7368
recall: 0.3022
f1: 0.4286
roc_auc: 0.8256

=== Ensemble Test Metrics ===
              precision    recall  f1-score   support

           0       0.84      0.97      0.90       528
           1       0.70      0.31      0.43       139

    accuracy                           0.83       667
   macro avg       0.77      0.64      0.66       667
weighted avg       0.81      0.83      0.80       667

threshold: 0.6000
accuracy: 0.8291
precision: 0.7049
recall: 0.3094
f1: 0.4300
roc_auc: 0.8354


In [14]:
test_result = merged_df.loc[X_test.index, ['channel_id', 'title']].copy()
test_result['churn_prob'] = ensemble_prob_test
test_result['predicted_churn'] = ensemble_pred
test_result['is_churned'] = y_test.values
test_result = test_result.sort_values('churn_prob', ascending=False).reset_index(drop=True)

print("=== Ensemble: Top 10 channels by churn probability ===")
print(test_result.head(10).to_string(index=False))


=== Ensemble: Top 10 channels by churn probability ===
              channel_id                       title  churn_prob  predicted_churn  is_churned
UCwTHUixtkE6UOZbGcAKfF6g            케이밥스타 [K-밥 STAR]    0.978166                1           1
UCbhdauhspaCEA7oy55STERw            Merry Play 메리플레이    0.976948                1           1
UCeFI3Mlzg4EmhEeSILLm69Q 슈퍼판도비 (Super Pandobi) -인기동요    0.957122                1           1
UCGNhjyXPRc3IX308JVxeZ0g               Chiro 치로와 친구들    0.947438                1           1
UCCJkwrmhIqWkSFV-sQol4Qw                 밍꼬발랄Mingggo    0.929501                1           1
UCyjw2Z7tC-iVNgELRxOwIgQ              하얀손 White hand    0.922624                1           1
UCCcE8BpgFqKRP3au8tEUrAw                 POWER MOVIE    0.890576                1           1
UCQ7X91NIBS174KJT4Id0lnQ           킬링벌's KillingBees    0.883529                1           1
UC5rYqOqmrUW9hMFPHtagKQA                        모노튜브    0.869589                1           1
UC06v

## 🎯 결론 및 모델 평가 (XGBoost + LightGBM 앙상블)

**1. 모델 구성 및 가중치**
* LightGBM (60%) + XGBoost (40%)
* 대표적인 2개의 그래디언트 부스팅(Gradient Boosting) 트리를 섞은 형태입니다.

**2. 앙상블 모델의 효용성 및 평가**
* 같은 부스팅 계열이므로 모델 간의 '다양성(Diversity)'은 RF를 섞었을 때보다 낮습니다. 즉, 비슷하게 맞추고 비슷하게 틀릴 확률이 높습니다.
* 하지만 두 강력한 알고리즘이 미세하게 놓치는 부분들을 서로 채워주어 최고 수준의 정확도를 낼 잠재력이 있습니다.

**3. 최종 활용 방안**
* 연산 리소스가 충분하고 오직 극단적인 정확도(Accuracy) 소폭 향상을 노릴 때 주로 캐글(Kaggle) 같은 대회에서 많이 쓰이는 방식입니다.
* 실무에서는 두 모델의 유지보수 비용 대비 얻을 수 있는 추가 이득이 크지 않을 수 있으므로, 굳이 2개를 써야 한다면 부스팅+배깅 조합(LGBM+RF)을 우선적으로 고려하는 것이 낫습니다.